In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta
import numpy as np

# 경로 설정
target_dir = "/mnt/AEGIS/AEGIS/Deidentification/AEGIS-Nursing-Initial-Assessment/de"

# 환자별 노이즈 정보
noise_df = pd.read_csv("/mnt/AEGIS/AEGIS/Deidentification/AEGIS-PT/pt_noise_date_time.csv", sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
noise_df['new_pt_id'] = noise_df['new_pt_id'].astype(str)

pt_id_map = noise_df.set_index('pt_id')['new_pt_id'].to_dict()
date_noise_map = noise_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_noise_map = noise_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(target_dir, '**/*.csv'), recursive = True)
print(f"총 처리 대상 파일 수 : {len(csv_files)}")



In [ ]:
# 컬럼 삭제
print("\n 컬럼 제거")

columns_to_drop = [
    'team', 'bed_num', 'cc_text', 'phx','travel_contry', 'etc_text', 'phx_text', 'surgical_history', 'protector', 'stuff',
    'lab_text', 'transfer_another','trauma', 'ed_ward', 'ad_ward_cd', 'transfer_text', 'dr_cc_cd', 'dr_cc', 'dr_cc_text', 'lmp',
    'edc', 'dced', 'gestation_week', 'gestation_day', 'amniotomy', 'amniotomy_dt', 'show', 'show_dt', 'parity_f', 'parity_p',
    'parity_a1', 'parity_a2', 'parity_l1', 'parity_l2', 'pre_delivery', 'menstrual_regular', 'menstrual_cycle', 'menstrual_from',
    'menstrual_to', 'fht1','fht2','fht3','fht4','cx_cm', 'cx_percent', 'fetal_anomaly', 'fetal_anomaly_result', 'fetal_anomaly_text',
    'mid_sono', 'mid_sono_result', 'mid_sono_text', 'gtt50', 'gtt50_result', 'gtt50_text', 'gtt100', 'gtt100_result', 'gtt100_text',
    'amniocentesis', 'amniocentesis_result', 'amniocentesis_text', 'mh_hiv', 'ex1_hiv', 'ex1_person_id','ex1_person', 
    'ex2_hiv', 'ex2_person_id', 'ex2_person', 'complication_text', 'nedis', 'nedis_ed', 'nedis_ad', 'trans_in_cd', 'trans_in',
    'nursing_home_num', 'nursing_home', 'trans_out_cd', 'trans_out', 'dc_trans_out_cd', 'dc_trans_out_nm'
]

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        total_cols_to_remove = len(columns_to_drop)
        cols_in_df = [col for col in columns_to_drop if col in df.columns]
        removed_cols_count = len(cols_in_df)
        
        if removed_cols_count > 0:
            df.drop(columns = cols_in_df, inplace = True)
            
        df.to_csv(file, index=False, encoding = 'utf-8')
        
        print(f"{file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")
    
    except Exception as e:
        print(f"[❗ hos_dt 오류] {file} → {e}")

In [ ]:
# opd_nm, depart, ad_ward에서 신부인과, 정신건강의학과 제거

import os
import glob
import pandas as pd


# 키워드 정의
keywords = "산부인과|정신건강의학과|정신의학과"

# 처리 시작
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        original_len = len(df)

        # 각 컬럼에 대해 조건에 맞는 행 제거
        condition = pd.Series([False] * len(df))
        for col in ['opd_nm', 'depart', 'ad_ward']:
            if col in df.columns:
                condition = condition | df[col].fillna('').str.contains(keywords, regex=True)

        # 해당 행 제거
        df = df[~condition]
        removed = original_len - len(df)

        # 덮어쓰기 저장
        df.to_csv(file, index=False, encoding='utf-8')
        print(f"✅ {os.path.basename(file)}: {removed}건 제거 완료")

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")


In [ ]:
# ① pt_id 치환
print("\n① pt_id 치환 시작")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')
        
        if 'pt_id' in df.columns:
            df['pt_id'] = df['pt_id'].map(pt_id_map).fillna(df['pt_id'])
            
        print(df.head(3))
        
        df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"✅ pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ pt_id 오류] {file} → {e}")

In [ ]:
# ② hos_dt ±7일 ±30분
print("\n② hos_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'hos_dt' in df.columns:
            df['hos_dt'] = pd.to_datetime(df['hos_dt'], errors='coerce')
            df['hos_dt'] = df['hos_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ hos_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ hos_dt 오류] {file} → {e}")



In [ ]:
# pre_tx_time ±7일 ±30분
print("\n② pre_tx_time 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'pre_tx_time' in df.columns:
            df['pre_tx_time'] = pd.to_datetime(df['pre_tx_time'], errors='coerce')
            df['pre_tx_time'] = df['pre_tx_time'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ pre_tx_time 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ pre_tx_time 오류] {file} → {e}")



In [ ]:
# os_dt ±7일 ±30분
print("\n os_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'os_dt' in df.columns:
            df['os_dt'] = pd.to_datetime(df['os_dt'], errors='coerce')
            df['os_dt'] = df['os_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ os_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ os_dt 오류] {file} → {e}")



In [ ]:
# f_ktas_dt ±7일 ±30분
print("\n f_ktas_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'f_ktas_dt' in df.columns:
            df['f_ktas_dt'] = pd.to_datetime(df['f_ktas_dt'], errors='coerce')
            df['f_ktas_dt'] = df['f_ktas_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ f_ktas_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ f_ktas_dt 오류] {file} → {e}")



In [ ]:
# l_ktas_dt ±7일 ±30분
print("\n l_ktas_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'l_ktas_dt' in df.columns:
            df['l_ktas_dt'] = pd.to_datetime(df['l_ktas_dt'], errors='coerce')
            df['l_ktas_dt'] = df['l_ktas_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ l_ktas_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ l_ktas_dt 오류] {file} → {e}")



In [ ]:
# ad_date 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        

        if 'ad_date' in df.columns:

            df['ad_date'] = pd.to_datetime(df['ad_date'], errors = 'coerce')

            

            df['ad_date'] = df['ad_date'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D")
            

            print(df.head(3))

            

            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"ad_date 완료: {file}")

            

    

    except Exception as e:

        print(f"[ad_date 오류] {file} -> {e}")

In [ ]:
# dc_dt ±7일 ±30분
print("\n dc_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'dc_dt' in df.columns:
            df['dc_dt'] = pd.to_datetime(df['dc_dt'], errors='coerce')
            df['dc_dt'] = df['dc_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df['dc_dt'])
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ dc_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ dc_dt 오류] {file} → {e}")



In [ ]:
# ex1_date 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        

        if 'ex1_date' in df.columns:

            df['ex1_date'] = pd.to_datetime(df['ex1_date'], errors = 'coerce')

            

            df['ex1_date'] = df['ex1_date'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D")
            

            print(df[~df['ex1_date'].isna()]['ex1_date'])

            

            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"ex1_date 완료: {file}")

            

    

    except Exception as e:

        print(f"[ex1_date 오류] {file} -> {e}")

In [ ]:
# ex2_date 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        

        if 'ex2_date' in df.columns:

            df['ex2_date'] = pd.to_datetime(df['ex2_date'], errors = 'coerce')

            

            df['ex2_date'] = df['ex2_date'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D")
            

            print(df[~df['ex2_date'].isna()]['ex2_date'])

            

            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"ex2_date 완료: {file}")

            

    

    except Exception as e:

        print(f"[ex2_date 오류] {file} -> {e}")

In [ ]:
# nedis_dt ±7일 ±30분
print("\n nedis_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'nedis_dt' in df.columns:
            df['nedis_dt'] = pd.to_datetime(df['nedis_dt'], errors='coerce')
            df['nedis_dt'] = df['nedis_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df['nedis_dt'])
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ nedis_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ nedis_dt 오류] {file} → {e}")



In [ ]:
# ad_certification_date ±7일 ±30분
print("\n ad_certification_date 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'ad_certification_date' in df.columns:
            df['ad_certification_date'] = pd.to_datetime(df['ad_certification_date'], errors='coerce')
            df['ad_certification_date'] = df['ad_certification_date'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df['ad_certification_date'])
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ ad_certification_date 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ ad_certification_date 오류] {file} → {e}")



In [ ]:
#  특이 정보 추출 (cc_cd, cc_nm N수 5 이하)

rare_combinations_all = []
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if {'cc_cd', 'cc_nm'}.issubset(df.columns):
            combo_counts = df.groupby(['cc_cd', 'cc_nm']).size().reset_index(name = 'count')
            rare_combos = combo_counts[combo_counts['count'] <= 5].copy()
            rare_combos['source_file'] = os.path.basename(file)
            rare_combinations_all.append(rare_combos)
            print(f"처리 완료: {file}")
        else:
            print(f"해당 컬럼 없음: {file}")
            
    except Exception as e:
        print(f"오류 발생: {file} -> {e}")
        
if rare_combinations_all:
    result_df = pd.concat(rare_combinations_all, ignore_index = True)
    result_df = result_df.sort_values(by='count', ascending = True).reset_index(drop = True)
    print(result_df.head())
else:
    print("조건에 맞는 조합 없음")
print(f"총 count 수: {result_df['count'].sum()}")

In [ ]:
# 특이 정보 제거 (cc_cd, cc_nm)

import os
import glob
import pandas as pd

# 희귀 조합 (result_df 기준)
rare_combos = result_df[['cc_cd', 'cc_nm']].dropna().drop_duplicates()

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id': str}, sep=',', quotechar='"', lineterminator='\n')

        if {'cc_cd', 'cc_nm'}.issubset(df.columns):
            before_rows = len(df)
            
            merged = df.merge(rare_combos, on = ['cc_cd', 'cc_nm'], how = 'left', indicator = True)
            df_filtered = merged[merged['_merge'] == 'left_only'].drop(columns = ['_merge'])
            
            after_rows = len(df_filtered)
            removed_count = before_rows - after_rows
            
            df_filtered.to_csv(file,index = False, encoding = 'utf-8')
            
            print(f"{file}: {removed_count}개 행 제거됨 (남은 행: {after_rows})")
            
        else:
            print(f"❗ 해당 컬럼 없음: {file}")

    except Exception as e:
        print(f"❌ 오류 발생: {file} → {e}")


In [ ]:
#  특이 정보 추출 (dig_nm N수 5 이하)

total_rare_rows = 0
all_rare_value_info = []

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'dig_nm' in df.columns:
            value_counts = df['dig_nm'].value_counts(dropna=True)
            
            rare_values = value_counts[value_counts <= 5]
            rare_value_list = rare_values.index.tolist()
            rare_count = df['dig_nm'].isin(rare_value_list).sum()
            
            df.loc[df['dig_nm'].isin(rare_value_list), 'dig_nm'] = 'Deidentified'           
            df.to_csv(file, index = False, encoding = 'utf-8')
            
            
            total_rare_rows += rare_count
            rare_info = pd.DataFrame({
                'source_file': os.path.basename(file),
                'dig_nm': rare_values.index,
                'count': rare_values.values
            })
            all_rare_value_info.append(rare_info)
            
            print(f"{os.path.basename(file)}: {rare_count}건 치환 완료, 희귀값 {len(rare_values)}개")
        else:
            print(f"해당 컬럼 없음: {file}")
            
    except Exception as e:
        print(f"오류 발생: {file} -> {e}")
    
if all_rare_value_info:
    if len(all_rare_value_info) == 1:
        rare_values_df = all_rare_value_info[0].copy()
    else:
        rare_values_df = pd.concat(all_rare_value_info, ignore_index = True)
    
    rare_values_df = rare_values_df.sort_values(by = 'count').reset_index(drop=True)
    
    print(f"\n 전체 희귀 dig_nm 값 치환된 총 행 수 : {len(rare_values_df)}건")
    print(rare_values_df.head(10))
else:
    print("희귀 dig_nm 값 없음")